In [1]:
import os
from rdflib import Graph
import pandas as pd
from IPython.display import display, Markdown

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1

INDEX_NAME = os.getenv("INDEX_NAME")
ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
FRAGMENT_INDEX_NAME = f"{INDEX_NAME}_fragments"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [60]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id
WHERE {{
    ?stmt a rdf:Statement ;
          rdf:subject ?s ;
          rdf:predicate ?p ;
          rdf:object ?o ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
    start_date=start_date,
    end_date=end_date,
    filtering_criteria=filtering_criteria
)   
    return graph.query(query)

In [15]:
def find_similar(term, val_type, score_threshold=0.5):
    if val_type not in ["entity", "predicate"]:
        raise ValueError("val_type must be either 'entity' or 'predicate'")

    query_embedding = EMBEDDING_MODEL.encode(term)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ENT_INDEX_NAME,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 20,
            "num_candidates": 100,
            "filter": {
                "match": {
                    "value_type": val_type
                }
            }
        },
        source=["value_name", "value_type"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"].replace(" ", "_"))

    return pd.DataFrame(final_res)

In [12]:
def get_fragment(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    start = int(row.start)
    end = int(row.end) + 1
    speech_id = row.speech_id
    return ".".join(es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"].split(".")[start:end])

def get_speech(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = row.speech_id - 1

    return es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [66]:
g = Graph()
g.parse(f"{INDEX_NAME}_fragments_2.ttl", format="turtle")

<Graph identifier=N44d2bcb1f3e44f78ba2768e524685d30 (<class 'rdflib.graph.Graph'>)>

In [67]:
QUESTION_ANSWER_PROMPT = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.
Each fact contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

Research question: {question}
Facts:
{facts}

Your task is to answer the research question based on the provided facts."
"""

In [69]:
sim_word = "threat"

similars = find_similar(sim_word, "entity", score_threshold=0.4)
similars

,score,value_name
0,1.000000,threat
1,0.888251,terrorist_threat
2,0.879718,threat_assessment
3,0.865144,future_threats
4,0.857553,modern_threats
5,0.857115,common_threats
6,0.856646,new_threats
7,0.837873,drug_threat
8,0.834585,threats_of_the_present_day
9,0.828622,attack


In [70]:
filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
triplets_resp = {
    "subject": [],
    "predicate": [],
    "object": [],
    "date": [],
    "start": [],
    "end": [],
    "speech_id": [],
}
for row in similars.itertuples(index=False):
    val = row.value_name
    query_res = execute_query(
        graph=g,
        start_date="1999-01-01",
        end_date="2024-12-31",
        filtering_criteria=filtering_criteria.format(val=val),
    )

    for row in query_res:
        fragment_start = row.start
        fragment_end = row.end
        speech_id = row.speech_id

        triplets_resp["subject"].append(row.s.split('/')[-1])
        triplets_resp["predicate"].append(row.p.split('/')[-1])
        triplets_resp["object"].append(row.o.split('/')[-1])
        triplets_resp["date"].append(str(row.date.value))
        triplets_resp["start"].append(fragment_start)
        triplets_resp["end"].append(fragment_end)
        triplets_resp["speech_id"].append(speech_id)
triplets_resp = pd.DataFrame(triplets_resp).reset_index().sort_values(by="date", ascending=True)
triplets_resp["index"] = triplets_resp.index + 1

In [71]:
QUESTION = """
How does he speak about “threats” to mobilize society?
"""

FACTS = triplets_resp[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")

PROMPT = QUESTION_ANSWER_PROMPT.format(question=QUESTION, facts=FACTS)
PROMPT

'\nYou are a political scientist model. You are given a research question and a set of facts from a knowledge graph.\nEach of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.\nEach of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.\nThe triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.\nEach fact contains an index.\nYou can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".\nUse just syntax with [index] to refer to the fact, do not use any additional words.\n\nResearch question: \nHow does he speak about “threats” to mobilize society?\n\nFacts:\n[{\'index\': 8, \'subject\': \'vlad

In [72]:
res = ollama_request(
    prompt=PROMPT, is_stream=False
)

display(Markdown(res))

Vladimir Putin frames **"threats"** as a strategic tool to mobilize society through several key approaches:

1. **Direct Warnings & Terrorism Focus** [8]
   He emphasizes terrorism resurgence (e.g., *"warns_of_terrorism_resurgence"*) to justify heightened security measures and public vigilance.

2. **Systematic Threat Assessment** [1, 9, 10]
   He repeatedly calls for structured threat evaluations (*"requires_threat_assessment_discussions"*, *"requires_closure_threat_assessment"*), framing threats as urgent priorities requiring collaboration (e.g., *"plans_collaborative_effort"*).

3. **Global Threats as Unifying Causes** [12, 16]
   He ties modern threats to global dynamics (*"impacts_worldwide"*, *"inspired_world_security_configuration"*), positioning them as existential risks that transcend borders.

4. **Regional & Future Risks** [11, 5]
   He highlights evolving threats (e.g., *"concerns_future_threats"*) and regional instability (*"may_modify_the_region"*), reinforcing a narrative of perpetual danger to justify long-term security policies.

5. **Multilateral Collaboration** [2, 3, 14]
   By linking threats to alliances (e.g., *"collaborates_against_threat"*, *"plans_joint_military_engagement"*), he mobilizes support for collective action against perceived adversaries.

6. **Diversified Threats** [19, 20]
   Beyond terrorism, he acknowledges other threats like drugs (*"recognizes_drug_threat"*) and current risks (*"mitigates_current_regional_risks"*), broadening the scope to justify comprehensive governance responses.

---
**Key Pattern**: Putin’s rhetoric on threats is **contextualized in geopolitical crises**, often tied to Russia’s strategic interests (e.g., NATO, Ukraine) while framing them as universal challenges. His approach combines **direct warnings** with **systematic threat management**, leveraging both domestic mobilization and international alliances.

In [73]:
REFERENCE_NUM = 6

display(triplets_resp[triplets_resp["index"] == REFERENCE_NUM][["subject", "predicate", "object"]].reset_index(drop=True))

display(Markdown("### Reference Fragment"))
display(Markdown(get_fragment(triplets_resp, REFERENCE_NUM)))

display(Markdown("### Reference Speech"))
display(Markdown(get_speech(triplets_resp, REFERENCE_NUM)))


,subject,predicate,object
0,wmd_proliferation,linked_by\nconnected_to,terrorist_threat


### Reference Fragment

 As for terminology, we are opposed to drawing up any blacklists. We proceed from the fact that the problems need to be dealt with. The problems are not only concentrated in the countries that you have mentioned. If we are talking about the main threat of the 21st century, I think that it is the problem of proliferation of weapons of mass destruction. And here, of course, we should not only mention North Korea, not only the Middle East, we should also mention South Asia. We should always remember that the problem of proliferation of nuclear and other weapons of mass destruction is closely related to another threat – the threat of terrorism, because terrorists attempt to acquire certain means of mass destruction. This is particularly dangerous

### Reference Speech

Good afternoon, Today we are to discuss the issues of security in the Siberian Federal District. In fact, we will have to take another look at the situation in the district as a whole. This is not our first meeting devoted to regional problems. The Security Council has already analysed the situation in the Far Eastern Federal District and in the Kaliningrad Region, and there is a consensus that it is a useful practice that enables us to look at the problems of the region in their entirety.  As regards the Siberian Federal District, I would like to mention the following main problems. First of all, about economics. The economic situation has stabilised somewhat in recent years, the gross regional product is growing, and industrial output grew by 4% last year.  But that is as much as I would like to say about the positive side, let us talk about the problems instead. Growth has mainly been achieved due to enterprises oriented towards commodity export. While labour in Siberia is short, its natural riches are colossal and it has major industrial and research centres and defence industries, but the yield from them is poor. Most Siberian regions survive on subsidies, their economies are based on raw materials extraction, they have run-down basic assets, are short of investments and have problems with replenishing their reserves of raw materials.  The situation is compounded by the high prices of energy and transport.  You know that the strategy of economic development in Siberia was adopted last year. Meanwhile the mechanisms of implementing it are not very clear, and no coherent decisions have been taken in pursuit of the initiatives that come from the district itself, notably regarding the increase of innovative activities. It would be beneficial to discuss the progress of these programmes today.  Another thing I would like to draw your attention to are the serious social problems. They are partly the result of the above mentioned economic imbalances. Over the past years, the incomes in Siberian regions have unfortunately been 20% lower than in Russia as a whole. The wage arrears are massive, and unemployment is higher than in Russia on average. The housing and utilities are in a sorry state. Sixty percent of the system is decrepit. All of this affects people’s lives, creates social tensions, leads to an outflow of qualified personnel and not only from enterprises, but also from government bodies. In the last 10 years, the population of the district has been shrinking at the rate of 100,000 a year. The trend is a serious threat to the region’s future.  Another important area of problems is the serious environmental and sanitary-epidemiological situation. Five of the ten environmentally unsafe Russian cities are in the Siberian District. Siberian enterprises account for 1/3 of harmful emissions. Forest fires have become a real disaster.  The death rate is very high, and the causes are social: drug addiction, tuberculosis, and AIDS. Drug addiction is 73% higher than the Russian average.  Special mention must be made of the military-technical security of our key strategic facilities. Their high concentration is a feature of the Siberian District. I would like you to dwell on this issue. Today let us discuss measures that need to be taken to improve that situation.  Furthermore. We still have been unable to reverse the negative crime trends. The crime level remains high, and serious and very serious crimes are predominant. Criminal groups try to gain control of key economic branches and natural resources, and the law enforcement bodies do not always counteract them effectively.  And finally, an important aspect of the situation in this particular region of Russia is border protection. The district has the country’s longest land border. It has yet to be properly developed. We must be honest with ourselves: we have not really come to grips with the problem of developing the border. Above all, that applies to the Russia-Kazakhstan and the Russia-Mongolia stretches. This is not only about protection against trans-border crime or creating a Soviet-style border strip. We are talking about creating a modern, civilised border that would set a barrier to illegal migration, contraband and trans-border crime while facilitating normal border cooperation, trade and tourism. In April of this year, we discussed these issues with the President of Kazakhstan and the heads of border regions. From the results of that meeting certain recommendations have been worked out and instructions have been given to the Government. Let us discuss what is being done to rectify the situation and to carry out these instructions. 